# Hands-on LoRA finetuning tutorial on Orange3 QA & MCQ dataset

### Preparing the environment

In [3]:
# @title 🛠️ Setup: Optional Clone & Install (commented out)
import os
import sys

# Optional: clone original tutorial repo (kept here for reference, commented out)
REPO_URL = "https://github.com/fulaibaowang/RAG-orange.git"
REPO_NAME = "RAG-orange"
branch = "main"

# if os.path.isdir(REPO_NAME):
#     print(f"🔄 Updating {REPO_NAME}...")
#     # !cd {REPO_NAME} && git pull origin {branch}
# else:
#     print(f"📥 Cloning {REPO_NAME}...")
#     # !git clone {REPO_URL}

# if REPO_NAME not in sys.path:
#     sys.path.append(os.path.abspath(REPO_NAME))

# Optional: install dependencies from the cloned repo (commented out)
# print("📦 Installing dependencies from cloned repo...")
#!pip install -q -r {REPO_NAME}/requirements.txt

print("Using local project files. External clone/install is kept above but commented out.")

Using local project files. External clone/install is kept above but commented out.


In [4]:
import os
import torch
import json
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel, IA3Config
from trl import SFTTrainer

import sys
sys.path.append('../src')
from store_load_results import store_results, load_results
from evaluation_function import evaluate_model
from utils import define_model_name

## PART 1: Basic Finetuning with LoRA using PEFT

### Defining the model config

In [5]:
MODEL_CONFIG = {
    "base_model": "Qwen/Qwen3-0.6B",
    "finetuning": True,
    "use_dora": True,
    "n_epochs": 1,
    "lora_r": 8,
    "lora_alpha": 16,
    "lr": 5e-4,
    "batch_size": 8,
    "lora_projections": "qvko",
    "lora_dropout": 0.05,
    "new_tokens_path": None,
    "new_tokens_init": "random",
    "new_tokens_train": True,
    "use_ia3": False
}

PROJECTIONS = {
    "q": "q_proj",
    "k": "k_proj",
    "v": "v_proj",
    "o": "o_proj",
    "g": "gate_proj",
    "d": "down_proj",
    "u": "up_proj"
}

projections = [PROJECTIONS[p] for p in list(MODEL_CONFIG["lora_projections"])]
MODEL_CONFIG['lora_projections'] = projections
model_name, OUTPUT_DIR = define_model_name(MODEL_CONFIG)

wandb_project = "qwen3-lora-finetuning"
wandb_run_name = model_name
os.environ["WANDB_PROJECT"] = wandb_project

print("Model configuration:")
for key, value in MODEL_CONFIG.items():
    print(f"   {key}: {value}")

Model Configuration: Qwen3-0.6B_DoRA_qvko_r8_alpha16_drop0.05_proj(qvko)_bs8_lr0.0005_ep1__
Model configuration:
   base_model: Qwen/Qwen3-0.6B
   finetuning: True
   use_dora: True
   n_epochs: 1
   lora_r: 8
   lora_alpha: 16
   lr: 0.0005
   batch_size: 8
   lora_projections: ['q_proj', 'v_proj', 'k_proj', 'o_proj']
   lora_dropout: 0.05
   new_tokens_path: None
   new_tokens_init: random
   new_tokens_train: True
   use_ia3: False
   model_name: Qwen3-0.6B_DoRA_qvko_r8_alpha16_drop0.05_proj(qvko)_bs8_lr0.0005_ep1__


## Training dataset

Let's look at the training dataset, and what kind of data we have.

In [6]:
if "__file__" in globals():
    PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'train_test_dataset')
TRAINDATA_FILE = os.path.join(DATA_DIR, 'orange_qa_train.jsonl')
TESTDATA_MCQ_FILE = os.path.join(DATA_DIR, 'orange_qa_MCQ_test.jsonl')
TESTDATA_MCQ_CON_FILE = os.path.join(DATA_DIR, 'orange_qa_MCQ-con_test.jsonl')

dataset = load_dataset("json", data_files=TRAINDATA_FILE, split="train")

Generating train split: 2906 examples [00:00, 54655.65 examples/s]


In [7]:
# check different question types:
# QA: 7
# MCQ: 579
# QA - connection: 0
# MCQ - conection: 1012

ID = 0
sample = dataset[ID]['messages']
for message in sample:
    print("Role:", message['role'])
    print(message['content'])
    print()

Role: system
You are a helpful assistant that can answer questions about the Orange Data Mining software.

Role: user
Answer the following question based on your knowledge of the Orange Data Mining software.
Make sure you answer the question in a few sentences.

Question: Am I correct to say the Distances widget is an invalid input to the Scatter Plot widget? Distances -x-> Scatter Plot


Role: assistant
Yes. The Distances widget is an invalid input to the Scatter Plot widget, notated as a connection Distances -x-> Scatter Plot.



### Load the model and tokenizer

Let's get familiar with the model and tokenizer. Specifically look at what layers the model has.

In [9]:
# 4. Load Model & Tokenizer
print("Loading model and tokenizer...")
#tokenizer = AutoTokenizer.from_pretrained(MODEL_CONFIG['base_model'])
tokenizer = AutoTokenizer.from_pretrained(MODEL_CONFIG['base_model'], padding_side="left")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_CONFIG['base_model'],
    dtype=torch.float16,       # Use float16 to save memory
    device_map="auto"          # Auto-selects GPU or CPU
)

# Baseline evaluation before LoRA fine-tuning
print("Running baseline evaluation on MCQ and MCQ-connection test sets before fine-tuning...")
with open(TESTDATA_MCQ_FILE, "r") as f:
    test_mcq_dataset = json.load(f)
accuracy_mcq_base, se_mcq_base = evaluate_model(model, tokenizer, test_mcq_dataset, batch_size=MODEL_CONFIG['batch_size'])

with open(TESTDATA_MCQ_CON_FILE, "r") as f:
    test_mcq_con_dataset = json.load(f)
accuracy_mcq_con_base, se_mcq_con_base = evaluate_model(model, tokenizer, test_mcq_con_dataset, batch_size=MODEL_CONFIG['batch_size'])

baseline_results = {
    "accuracy_mcq": accuracy_mcq_base,
    "se_mcq": se_mcq_base,
    "accuracy_mcq_con": accuracy_mcq_con_base,
    "se_mcq_con": se_mcq_con_base,
}
print("Baseline Evaluation Results:", baseline_results)
baseline_config = {**MODEL_CONFIG, "finetuning": False}
store_results(baseline_results, baseline_config)
print("Baseline results stored successfully.")

Loading model and tokenizer...


Loading weights: 100%|█| 311/311 [00:01<00:00, 165.68it/s
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Running baseline evaluation on MCQ and MCQ-connection test sets before fine-tuning...


Evaluating model: 100%|██| 25/25 [00:52<00:00,  2.10s/it]

Baseline Evaluation Results: {'accuracy_mcq': np.float64(5.0), 'se_mcq': np.float64(1.5411035007422442), 'accuracy_mcq_con': np.float64(13.5), 'se_mcq_con': np.float64(2.4163505540380514)}
Baseline results stored successfully.


In [7]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [8]:
tokenizer

Qwen2Tokenizer(name_or_path='Qwen/Qwen3-0.6B', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=False, lstrip

In [9]:
# "Orange data mining", "widget", "Hierarchical Clustering", "-->", "-x->"
tokenizer(["Orange Data Mining", "orange data mining"])

{'input_ids': [[41969, 2885, 25832], [34164, 821, 11673]], 'attention_mask': [[1, 1, 1], [1, 1, 1]]}

### PEFT Module for LoRA/DoRA

In [10]:
peft_config = LoraConfig(
    bias="none",
    task_type="CAUSAL_LM",
    # from config
    target_modules=MODEL_CONFIG['lora_projections'],
    use_dora=MODEL_CONFIG['use_dora'],
    r=MODEL_CONFIG['lora_r'],
    lora_alpha=MODEL_CONFIG['lora_alpha'],
    lora_dropout=MODEL_CONFIG['lora_dropout'],
    modules_to_save=None,
)
if MODEL_CONFIG['use_ia3']:
    peft_config = IA3Config(
        peft_type="IA3",
        task_type="CAUSAL_LM",
        target_modules=MODEL_CONFIG['lora_projections'],
    )
    MODEL_CONFIG['model_name'] = MODEL_CONFIG['model_name'].replace("LoRA", "IA3").replace("DoRA", "IA3")
model = get_peft_model(model, peft_config)

In [11]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict(
                  (default): lora.dora.DoraLinearLayer

In [12]:
for name, param in model.base_model.model.model.layers[0].self_attn.named_parameters():
    print(name, param.requires_grad)

q_proj.base_layer.weight False
q_proj.lora_A.default.weight True
q_proj.lora_B.default.weight True
q_proj.lora_magnitude_vector.default.weight True
k_proj.base_layer.weight False
k_proj.lora_A.default.weight True
k_proj.lora_B.default.weight True
k_proj.lora_magnitude_vector.default.weight True
v_proj.base_layer.weight False
v_proj.lora_A.default.weight True
v_proj.lora_B.default.weight True
v_proj.lora_magnitude_vector.default.weight True
o_proj.base_layer.weight False
o_proj.lora_A.default.weight True
o_proj.lora_B.default.weight True
o_proj.lora_magnitude_vector.default.weight True
q_norm.weight False
k_norm.weight False


## Setup training arguments and Train

In [13]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=MODEL_CONFIG['n_epochs'],          # How many times to read the docs
    per_device_train_batch_size=MODEL_CONFIG['batch_size'],
    gradient_accumulation_steps=1,
    learning_rate=MODEL_CONFIG['lr'],
    fp16=True,                   # Use mixed precision
    logging_steps=10,
    optim="adamw_torch",
    save_strategy="epoch",       # Save a checkpoint every epoch
    report_to=["wandb"],         # Enable wandb logging
    run_name=wandb_run_name,     # Set run name to model name
)

# 8. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/2906 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2906 [00:00<?, ? examples/s]

In [14]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,2.283908
20,0.916111
30,0.730320
40,2.457719
50,2.673683
60,0.602405
70,0.576740
80,0.394542
90,0.436321
100,0.468020


wandb: WARNING URL not available in offline run


TrainOutput(global_step=364, training_loss=0.6151410216813559, metrics={'train_runtime': 333.8645, 'train_samples_per_second': 8.704, 'train_steps_per_second': 1.09, 'total_flos': 1296473747374080.0, 'train_loss': 0.6151410216813559})

### Evaluate the model on MCQ and MCQ-connection test sets

In [15]:
# 12. Evaluate and store results
with open(TESTDATA_MCQ_FILE, "r") as f:
    test_mcq_dataset = json.load(f)
accuracy_mcq, se_mcq = evaluate_model(model, tokenizer, test_mcq_dataset, batch_size=MODEL_CONFIG['batch_size'])

with open(TESTDATA_MCQ_CON_FILE, "r") as f:
    test_mcq_con_dataset = json.load(f)
accuracy_mcq_con, se_mcq_con = evaluate_model(model, tokenizer, test_mcq_con_dataset, batch_size=MODEL_CONFIG['batch_size'])
results = {
    "accuracy_mcq": accuracy_mcq,
    "se_mcq": se_mcq,
    "accuracy_mcq_con": accuracy_mcq_con,
    "se_mcq_con": se_mcq_con,
}
print("Evaluation Results:", results)
store_results(results, MODEL_CONFIG)
print("Results stored successfully.")

Evaluating model: 100%|██████████| 25/25 [02:04<00:00,  5.00s/it]

Evaluation Results: {'accuracy_mcq': np.float64(64.0), 'se_mcq': np.float64(3.394112549695428), 'accuracy_mcq_con': np.float64(13.5), 'se_mcq_con': np.float64(2.4163505540380514)}
Results stored successfully.


## PART 2: Token-injection of Orange3 widgets

If you are running out of GPU memory, you can restart the session and run the following cell again.

Just make sure to also run cells from here onwards.

In [16]:
## Delete if still in memory to free up space
del model
del tokenizer
torch.cuda.empty_cache()

In [17]:
import os
import torch
import json
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

import sys
sys.path.append('LoRA-finetuning-tutorial/src')
sys.path.append('../src')
from store_load_results import store_results, load_results
from evaluation_function import evaluate_model
from utils import define_model_name

In [18]:
MODEL_CONFIG = {
    "base_model": "Qwen/Qwen3-0.6B",
    "finetuning": True,
    "use_dora": True,
    "n_epochs": 1,
    "lora_r": 8,
    "lora_alpha": 16,
    "lr": 5e-4,
    "batch_size": 8,
    "lora_projections": "qvko",
    "lora_dropout": 0.05,
    "new_tokens_path": "LoRA-finetuning-tutorial/data/injected_tokens.json", ## path to new tokens file
    "new_tokens_init": "random",
    "new_tokens_train": True,
}

PROJECTIONS = {
    "q": "q_proj",
    "k": "k_proj",
    "v": "v_proj",
    "o": "o_proj",
    "g": "gate_proj",
    "d": "down_proj",
    "u": "up_proj"
}

projections = [PROJECTIONS[p] for p in list(MODEL_CONFIG["lora_projections"])]
MODEL_CONFIG['lora_projections'] = projections
model_name, OUTPUT_DIR = define_model_name(MODEL_CONFIG)

wandb_project = "qwen3-lora-finetuning"
wandb_run_name = model_name
os.environ["WANDB_PROJECT"] = wandb_project

print("Model configuration:")
for key, value in MODEL_CONFIG.items():
    print(f"   {key}: {value}")

Model Configuration: Qwen3-0.6B_DoRA_qvko_r8_alpha16_drop0.05_proj(qvko)_bs8_lr0.0005_ep1_ntinit(random)_nttrain
Model configuration:
   base_model: Qwen/Qwen3-0.6B
   finetuning: True
   use_dora: True
   n_epochs: 1
   lora_r: 8
   lora_alpha: 16
   lr: 0.0005
   batch_size: 8
   lora_projections: ['q_proj', 'v_proj', 'k_proj', 'o_proj']
   lora_dropout: 0.05
   new_tokens_path: LoRA-finetuning-tutorial/data/injected_tokens.json
   new_tokens_init: random
   new_tokens_train: True
   model_name: Qwen3-0.6B_DoRA_qvko_r8_alpha16_drop0.05_proj(qvko)_bs8_lr0.0005_ep1_ntinit(random)_nttrain


In [19]:
if "__file__" in globals():
    PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'train_test_dataset')
TRAINDATA_FILE = os.path.join(DATA_DIR, 'orange_qa_train.jsonl')
TESTDATA_MCQ_FILE = os.path.join(DATA_DIR, 'orange_qa_MCQ_test.jsonl')
TESTDATA_MCQ_CON_FILE = os.path.join(DATA_DIR, 'orange_qa_MCQ-con_test.jsonl')

dataset = load_dataset("json", data_files=TRAINDATA_FILE, split="train")

In [23]:
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CONFIG['base_model'])
model = AutoModelForCausalLM.from_pretrained(
    MODEL_CONFIG['base_model'],
    dtype=torch.float16,       # Use float16 to save memory
    device_map="auto",          # Auto-selects GPU or CPU
)

Loading model and tokenizer...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


### Lets check the tokenizer again, and how it handles some common tokens

In [24]:
tokenizer

Qwen2Tokenizer(name_or_path='Qwen/Qwen3-0.6B', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=False, lstrip

In [25]:
tokenizer(["Hierarchical Clustering", "-->", "-x->", "Logistic Regression", "Hierarchical Clustering -x-> Logistic Regression"])

{'input_ids': [[74909, 45234, 2435, 36694], [29052], [6558, 405], [2201, 4532, 47470], [74909, 45234, 2435, 36694, 481, 87, 405, 78068, 47470]], 'attention_mask': [[1, 1, 1, 1], [1], [1, 1], [1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1]]}

### Adding new tokens to tokenizer & Extending model embeddings

In [26]:
from transformers import AddedToken

# 5. Extend token library (tokenizer + model)
if MODEL_CONFIG['new_tokens_path'] is not None:
    print("Adding new tokens from:", MODEL_CONFIG['new_tokens_path'])
    DEFAULT_TOKEN_NUM = len(tokenizer)

    with open(MODEL_CONFIG['new_tokens_path'], 'r') as f:
        new_tokens_text = json.load(f)
    new_token_tokenized = [tokenizer.encode(token) for token in new_tokens_text]
    new_tokens = [
        AddedToken(token, lstrip=True, rstrip=True)
        for token in new_tokens_text
    ]

    ## extend tokenizer and model
    print("Number of default tokenizer tokens:", DEFAULT_TOKEN_NUM)
    tokenizer.add_tokens(new_tokens)
    model.resize_token_embeddings(len(tokenizer))
    print("Number of new tokenizer tokens:", len(tokenizer))

    new_tokens_ids = [
        tokenizer.encode(token)[0]
        for token in new_tokens_text
    ]
    old_tokens_ids = [id for id in range(DEFAULT_TOKEN_NUM) if id not in new_tokens_ids]

    ## initialize new token embeddings
    model.model.embed_tokens = model.model.embed_tokens.float()

    if MODEL_CONFIG['new_tokens_init'] == "average":
        with torch.no_grad():
            existing_embeddings = model.model.embed_tokens.weight
            for new_id, previous_token_ids in zip(new_tokens_ids, new_token_tokenized):
                model.model.embed_tokens.weight[new_id, :] = existing_embeddings[previous_token_ids].mean(dim=0)
    elif MODEL_CONFIG['new_tokens_init'] == "zero":
        with torch.no_grad():
            model.model.embed_tokens[DEFAULT_TOKEN_NUM:, :].fill_(0.0)
    elif MODEL_CONFIG['new_tokens_init'] == "random":
        pass  # already quazi randomly initialized
    else:
        raise ValueError(f"Unknown new_tokens_init method: {MODEL_CONFIG['new_tokens_init']}")

    ## setup trainable weights for new tokens
    if MODEL_CONFIG['new_tokens_train']:
        def zero_out_old_token_grads(grad):
            new_grad = grad.clone()
            new_grad[old_tokens_ids, :] = 0.0
            return new_grad
        model.model.embed_tokens.weight.requires_grad = True
        model.model.embed_tokens.weight.register_hook(zero_out_old_token_grads)
    model.lm_head.weight.requires_grad = True

Adding new tokens from: LoRA-finetuning-tutorial/data/injected_tokens.json
Number of default tokenizer tokens: 151669
Number of new tokenizer tokens: 151751


### Let's test our tokenizer with new tokens

In [27]:
tokenizer

Qwen2Tokenizer(name_or_path='Qwen/Qwen3-0.6B', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	1703: AddedToken("File", rstrip=True, lstrip=True, single_word=False, normalized=True, special=False),
	5632: AddedToken("Filter", rstrip=True, lstrip=True, single_word=False, normalized=True, special=False),
	6533: AddedToken("Tree", rstrip=True, lstrip=True, single_word=False, normalized=True, special=False),
	22550: AddedToken("Rank", rstrip=True, lstrip=True, single_word=False, normalized=True, special=False),
	24862: AddedToken("Twitter", rstrip=True, lstrip=True, single_word=False, normalized=True, special=False),
	29052: AddedToken("-->", rstrip=True, lstrip=True, single_word=False, normalized=True, special=False),
	62707: AddedToken("Difference", rstrip=True, lstrip=True, single_word=False, normalized=True, special=False),
	81450: AddedToken("PCA

In [28]:
tokenizer(["Hierarchical Clustering", "-->", "-x->", "Logistic Regression", "Hierarchical Clustering -x-> Logistic Regression"])

{'input_ids': [[151744], [29052], [151750], [151711], [151744, 151750, 151711]], 'attention_mask': [[1], [1], [1], [1], [1, 1, 1]]}

In [29]:
if MODEL_CONFIG['new_tokens_path'] is not None:
    modules_to_save = ["lm_head"] + (["embed_tokens"] if MODEL_CONFIG['new_tokens_train'] else [])
else:
    modules_to_save = None

peft_config = LoraConfig(
    bias="none",
    task_type="CAUSAL_LM",
    # from config
    target_modules=MODEL_CONFIG['lora_projections'],
    use_dora=MODEL_CONFIG['use_dora'],
    r=MODEL_CONFIG['lora_r'],
    lora_alpha=MODEL_CONFIG['lora_alpha'],
    lora_dropout=MODEL_CONFIG['lora_dropout'],
    modules_to_save=modules_to_save,
)
model = get_peft_model(model, peft_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


In [30]:
print("LM head module requires grad: ", model.base_model.model.lm_head.modules_to_save.default.weight.requires_grad)
print("Embed tokens module requires grad: ", model.base_model.model.model.embed_tokens.modules_to_save.default.weight.requires_grad)

LM head module requires grad:  True
Embed tokens module requires grad:  True


In [31]:
wandb_project = MODEL_CONFIG.get('wandb_project') or os.environ.get("WANDB_PROJECT", "qwen3-lora-finetuning")
wandb_run_name = MODEL_CONFIG['model_name']

# Set wandb project via environment variable
os.environ["WANDB_PROJECT"] = wandb_project

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=MODEL_CONFIG['n_epochs'],          # How many times to read the docs
    per_device_train_batch_size=MODEL_CONFIG['batch_size'],
    gradient_accumulation_steps=1,
    learning_rate=MODEL_CONFIG['lr'],
    fp16=True,                   # Use mixed precision
    logging_steps=10,
    optim="adamw_torch",
    save_strategy="epoch",       # Save a checkpoint every epoch
    report_to=["wandb"],         # Enable wandb logging
    run_name=wandb_run_name,     # Set run name to model name
)

# 8. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/2906 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2906 [00:00<?, ? examples/s]

In [32]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.286709
20,0.785186
30,0.684635
40,0.614495
50,0.609649
60,0.584738
70,0.632727
80,0.469801
90,0.497010
100,0.536934


wandb: WARNING URL not available in offline run
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=364, training_loss=0.5151913378919873, metrics={'train_runtime': 566.015, 'train_samples_per_second': 5.134, 'train_steps_per_second': 0.643, 'total_flos': 1204811022643200.0, 'train_loss': 0.5151913378919873})

In [33]:
## Training in FP32, now we have to cast back to FP16 to evaluate
model.base_model.model.model.embed_tokens.modules_to_save.default = model.base_model.model.model.embed_tokens.modules_to_save.default.half()
model.base_model.model.lm_head.modules_to_save.default = model.base_model.model.lm_head.modules_to_save.default.half()

In [34]:
# 12. Evaluate and store results
with open(TESTDATA_MCQ_FILE, "r") as f:
    test_mcq_dataset = json.load(f)
accuracy_mcq, se_mcq = evaluate_model(model, tokenizer, test_mcq_dataset, batch_size=MODEL_CONFIG['batch_size'])

with open(TESTDATA_MCQ_CON_FILE, "r") as f:
    test_mcq_con_dataset = json.load(f)
accuracy_mcq_con, se_mcq_con = evaluate_model(model, tokenizer, test_mcq_con_dataset, batch_size=MODEL_CONFIG['batch_size'])
results = {
    "accuracy_mcq": accuracy_mcq,
    "se_mcq": se_mcq,
    "accuracy_mcq_con": accuracy_mcq_con,
    "se_mcq_con": se_mcq_con,
}
print("\nEvaluation Results:", results)
store_results(results, MODEL_CONFIG)
print("Results stored successfully.")

Evaluating model: 100%|██████████| 25/25 [00:22<00:00,  1.09it/s]


Evaluation Results: {'accuracy_mcq': np.float64(60.0), 'se_mcq': np.float64(3.4641016151377544), 'accuracy_mcq_con': np.float64(17.0), 'se_mcq_con': np.float64(2.656124997058685)}
Results stored successfully.
